In [2]:
import pandas as pd

df = pd.read_csv('../data/interim/monthly_panel.csv')
df['month'] = pd.to_datetime(df['month'])

expected_range = pd.date_range(start='2019-01-01', end='2024-12-01', freq='MS')

tags = df['tag'].unique()
missingness_results = []

for tag in tags:
    tag_df = df[df['tag'] == tag]
    actual_months = tag_df['month']
    
    missing = expected_range[~expected_range.isin(actual_months)]
    
    missingness_results.append({
        'Tag': tag,
        'Group': tag_df['group'].iloc[0],
        'Months Found': len(actual_months),
        'Months Missing': len(missing),
        'Missing List': missing.strftime('%Y-%m').tolist()
    })

report_df = pd.DataFrame(missingness_results)
print(report_df[['Tag', 'Group', 'Months Found', 'Months Missing']])

dupes = df.duplicated(subset=['month', 'tag']).sum()
print(f"\nDuplicate rows found: {dupes}")

           Tag           Group  Months Found  Months Missing
0     assembly  llm_unfriendly            72               0
1        cobol  llm_unfriendly            72               0
2          css    llm_friendly            72               0
3     embedded  llm_unfriendly            72               0
4      fortran  llm_unfriendly            72               0
5         html    llm_friendly            72               0
6   javascript    llm_friendly            72               0
7       kernel  llm_unfriendly            72               0
8       pandas    llm_friendly            72               0
9       python    llm_friendly            72               0
10     reactjs    llm_friendly            72               0
11         sql    llm_friendly            72               0
12     verilog  llm_unfriendly            72               0
13        vhdl  llm_unfriendly            71               1

Duplicate rows found: 0


In [3]:
df['z_score'] = df.groupby('tag')['n_questions'].transform(lambda x: (x - x.mean()) / x.std())

outliers = df[abs(df['z_score']) > 3]
print(f"Number of statistical outliers found: {len(outliers)}")
print(outliers[['month', 'tag', 'n_questions', 'z_score']])

Number of statistical outliers found: 2
         month      tag  n_questions   z_score
880 2020-05-01  verilog           89  4.181326
940 2019-05-01     vhdl           60  3.698113


In [4]:
negative_checks = (df[['n_questions', 'avg_score', 'avg_body_len']] < 0).any().any()

rate_checks = (df[['accept_rate', 'answered_rate']] > 1.0).any().any()

zero_len_check = ((df['n_questions'] > 0) & (df['avg_body_len'] == 0)).any()

print(f"Any negative values? {negative_checks}")
print(f"Any rates > 100%? {rate_checks}")
print(f"Any zero-length questions? {zero_len_check}")

Any negative values? True
Any rates > 100%? False
Any zero-length questions? False


In [5]:
logic_fail = df[df['unique_askers'] > df['n_questions']]
print(f"Logic failures found: {len(logic_fail)}")

Logic failures found: 0
